<a href="https://colab.research.google.com/github/asmitsingh1626/WEB_DEVELOPMENT/blob/main/Asmit_2026b03110004.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 24: Mobile App Usage & User Engagement Analysis

## Setup
Column names match your dataset (1,015 rows). Keep the file in the same folder as this notebook.

In [24]:
# If you get "No module named openpyxl", run once:  !pip install openpyxl
import os, glob
import pandas as pd
import numpy as np

# The notebook finds your data file automatically (xlsx or csv in this folder).
# To force a file, set FILE = "your_file_name.xlsx"
FILE = None
if FILE is None:
    candidates = (glob.glob("*mobile_app_usage*.xlsx") + glob.glob("*mobile_app_usage*.csv")
                  + glob.glob("*.xlsx") + glob.glob("*.csv"))
    if not candidates:
        raise FileNotFoundError("Put the data file in the same folder as this notebook. Current folder: " + os.getcwd())
    FILE = candidates[0]
print("Using file:", FILE)
COL_SESSION = "Session_ID"
COL_USER = "User_ID"
COL_DATE = "Session_Date"
COL_DUR = "Session_Duration_Mins"
COL_RATING = "User_Rating"
COL_FEATURE = "Feature_Used"
COL_DEVICE = "Device_Type"
COL_AGE = "Age_Group"

Using file: 24_mobile_app_usage_extended (4).xlsx


## 1. Load and inspect the dataset

In [25]:
df = pd.read_csv(FILE) if FILE.lower().endswith(".csv") else pd.read_excel(FILE)
df.columns = df.columns.str.strip()   # remove accidental spaces in headers
df.head()

,Session_ID,User_ID,Session_Date,Feature_Used,Session_Duration_Mins,User_Rating,Device_Type,Age_Group,App_Version,Data_Source
0,SE000000,USR00198,2025-04-29 08:00:00,Video,39.1,4.6,Android,18-24,1.2,Original
1,SE000001,USR00197,2026-03-02 02:00:00,Shopping,55.1,2.9,Android,18-24,2.0,Original
2,SE000002,USR00126,2026-04-26 09:00:00,Video,40.7,4.4,iOS,35-44,1.2,Original
3,SE000003,USR00398,2026-08-21 13:00:00,Search,14.6,3.3,Android,25-34,2.0,Original
4,SE000004,USR00105,2025-10-09 13:00:00,Messaging,39.6,1.1,Android,55+,1.2,Original


In [26]:
print(df.shape)
df.info()

(1315, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1315 entries, 0 to 1314
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Session_ID             1315 non-null   object        
 1   User_ID                1315 non-null   object        
 2   Session_Date           1315 non-null   datetime64[ns]
 3   Feature_Used           1315 non-null   object        
 4   Session_Duration_Mins  1282 non-null   float64       
 5   User_Rating            1281 non-null   float64       
 6   Device_Type            1315 non-null   object        
 7   Age_Group              1315 non-null   object        
 8   App_Version            1315 non-null   float64       
 9   Data_Source            1315 non-null   object        
dtypes: datetime64[ns](1), float64(3), object(6)
memory usage: 102.9+ KB


In [27]:
df.describe(include="all")

,Session_ID,User_ID,Session_Date,Feature_Used,Session_Duration_Mins,User_Rating,Device_Type,Age_Group,App_Version,Data_Source
count,1315,1315,1315,1315,1282.000000,1281.000000,1315,1315,1315.000000,1315
unique,1300,533,NaN,8,NaN,NaN,3,5,NaN,2
top,SE000737,USR00398,NaN,Shopping,NaN,NaN,iOS,25-34,NaN,Original
freq,2,8,NaN,182,NaN,NaN,446,275,NaN,1015
mean,NaN,NaN,2025-11-29 00:04:52.927756544,NaN,62.139704,3.091647,NaN,NaN,1.482510,NaN
min,NaN,NaN,2025-01-01 02:00:00,NaN,1.000000,1.000000,NaN,NaN,1.000000,NaN
25%,NaN,NaN,2025-06-13 11:00:00,NaN,31.825000,2.200000,NaN,NaN,1.100000,NaN
50%,NaN,NaN,2025-12-03 11:00:00,NaN,63.050000,3.100000,NaN,NaN,1.200000,NaN
75%,NaN,NaN,2026-05-07 14:00:00,NaN,91.675000,4.100000,NaN,NaN,2.000000,NaN
max,NaN,NaN,2026-10-31 15:00:00,NaN,120.700000,5.000000,NaN,NaN,2.100000,NaN


## 2. Check missing values and duplicate Session_IDs

In [28]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate Session_IDs:", df.duplicated(subset=COL_SESSION).sum())

Missing values:
 Session_ID                0
User_ID                   0
Session_Date              0
Feature_Used              0
Session_Duration_Mins    33
User_Rating              34
Device_Type               0
Age_Group                 0
App_Version               0
Data_Source               0
dtype: int64

Duplicate Session_IDs: 15


In [29]:
df = df.drop_duplicates(subset=COL_SESSION, keep="first")
df.shape

(1300, 10)

## 3. Handle missing Session_Duration_Mins / User_Rating
**Justification:** durations are right-skewed, so the median is safer than the mean. Ratings are bounded/ordinal, so the median avoids inventing fractional ratings.

In [30]:
df[COL_DUR] = df[COL_DUR].fillna(df[COL_DUR].median())
df[COL_RATING] = df[COL_RATING].fillna(df[COL_RATING].median())
df[[COL_DUR, COL_RATING]].isnull().sum()

,0
Session_Duration_Mins,0
User_Rating,0


## 4. Convert Session_Date to datetime; extract month, weekday, hour

In [31]:
df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce")
df["Month"] = df[COL_DATE].dt.month_name()
df["Weekday"] = df[COL_DATE].dt.day_name()
df["Hour"] = df[COL_DATE].dt.hour
df[[COL_DATE, "Month", "Weekday", "Hour"]].head()

,Session_Date,Month,Weekday,Hour
0,2025-04-29 08:00:00,April,Tuesday,8
1,2026-03-02 02:00:00,March,Monday,2
2,2026-04-26 09:00:00,April,Sunday,9
3,2026-08-21 13:00:00,August,Friday,13
4,2025-10-09 13:00:00,October,Thursday,13


## 5. Total sessions, unique users, average session duration

In [32]:
total_sessions = len(df)
unique_users = df[COL_USER].nunique()
avg_duration = df[COL_DUR].mean()
print(f"Total sessions: {total_sessions}")
print(f"Unique users: {unique_users}")
print(f"Average session duration: {avg_duration:.2f} mins")

Total sessions: 1300
Unique users: 533
Average session duration: 62.08 mins


## 6. Top 10 most frequently used features

In [33]:
feature_table = df[COL_FEATURE].value_counts().head(10).rename("Sessions")
feature_table

,Sessions
Feature_Used,
Shopping,179
Messaging,173
Video,167
Maps,165
Search,159
Profile,159
News,149
Payments,149


## 7. Average session duration by device type

In [34]:
device_comp = df.groupby(COL_DEVICE)[COL_DUR].agg(["mean", "count"]).sort_values("mean", ascending=False)
device_comp

,mean,count
Device_Type,,
iOS,64.286907,443
Android,62.487089,426
Web,59.400000,431


## 8. Rank age groups by total sessions

In [35]:
age_rank = df.groupby(COL_AGE)[COL_SESSION].count().sort_values(ascending=False)
age_rank

,Session_ID
Age_Group,
25-34,273
18-24,268
35-44,266
55+,248
45-54,245


## 9. Average user rating by feature

In [36]:
feature_rating = df.groupby(COL_FEATURE)[COL_RATING].mean().sort_values(ascending=False)
feature_rating

,User_Rating
Feature_Used,
Payments,3.261745
Video,3.244311
Search,3.145283
Shopping,3.075978
Profile,3.034591
Maps,3.033333
News,2.995973
Messaging,2.938728


## 10. Peak usage hours and busiest weekdays

In [37]:
peak_hours = df["Hour"].value_counts().head(5)
busiest_days = df["Weekday"].value_counts()
print("Peak hours:\n", peak_hours)
print("\nBusiest weekdays:\n", busiest_days)

Peak hours:
 Hour
8     82
1     69
2     62
18    60
20    60
Name: count, dtype: int64

Busiest weekdays:
 Weekday
Wednesday    198
Sunday       192
Friday       190
Tuesday      183
Thursday     182
Saturday     182
Monday       173
Name: count, dtype: int64


## 11. List for most-used features, tuple for device categories

In [38]:
top_features_list = feature_table.index.tolist()            # list
device_categories = tuple(sorted(df[COL_DEVICE].unique()))  # tuple (fixed)
print(top_features_list)
print(device_categories)

['Shopping', 'Messaging', 'Video', 'Maps', 'Search', 'Profile', 'News', 'Payments']
('Android', 'Web', 'iOS')


## 12. Set of unique users and a function for average duration of a feature

In [39]:
unique_user_set = set(df[COL_USER])
print("Unique users (set):", len(unique_user_set))

Unique users (set): 533


In [40]:
def avg_duration_for_feature(data, feature):
    """Average session duration for a selected feature."""
    subset = data[data[COL_FEATURE] == feature]
    return subset[COL_DUR].mean() if not subset.empty else float("nan")

sel = top_features_list[0]
print(f"Avg duration for '{sel}': {avg_duration_for_feature(df, sel):.2f} mins")

Avg duration for 'Shopping': 62.58 mins


## 13. NumPy: mean, median, standard deviation of session duration

In [41]:
dur = df[COL_DUR].to_numpy()
print(f"Mean: {np.mean(dur):.2f}")
print(f"Median: {np.median(dur):.2f}")
print(f"Std dev: {np.std(dur, ddof=1):.2f}")

Mean: 62.08
Median: 63.00
Std dev: 33.52


## 14. IQR outliers and engagement labels
**Rules:** Low < Q1, Medium = Q1 to Q3, High > Q3. Outliers lie outside Q1 − 1.5·IQR and Q3 + 1.5·IQR.

In [42]:
q1, q3 = np.percentile(dur, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df["Is_Outlier"] = (df[COL_DUR] < lower) | (df[COL_DUR] > upper)
print(f"IQR bounds: {lower:.2f} to {upper:.2f}")
print("Unusually long sessions:", (df[COL_DUR] > upper).sum())

IQR bounds: -55.89 to 179.21
Unusually long sessions: 0


In [43]:
def engagement(d):
    if d < q1:
        return "Low"
    elif d <= q3:
        return "Medium"
    return "High"

df["Engagement"] = df[COL_DUR].apply(engagement)
df["Engagement"].value_counts()

,count
Engagement,
Medium,650
Low,325
High,325


## 15. AppUser class and five recommendations

In [44]:
class AppUser:
    def __init__(self, user_id, device_type):
        self.user_id = user_id
        self.device_type = device_type

    def __repr__(self):
        return f"AppUser(user_id={self.user_id}, device_type={self.device_type})"

row = df.iloc[0]
AppUser(row[COL_USER], row[COL_DEVICE])

AppUser(user_id=USR00198, device_type=Android)

In [45]:
recommendations = [
    f"Promote and improve '{top_features_list[0]}', the most used feature, and surface lower-used features in onboarding.",
    f"Schedule push notifications around peak hours ({', '.join(map(str, peak_hours.index[:3]))}).",
    f"Target the least active age group ('{age_rank.index[-1]}') with tailored content or incentives.",
    f"Fix UX issues on '{feature_rating.index[-1]}', the lowest-rated feature.",
    f"Optimise the experience for '{device_comp.index[-1]}' devices, which have the shortest sessions.",
]
for i, r in enumerate(recommendations, 1):
    print(f"{i}. {r}")

1. Promote and improve 'Shopping', the most used feature, and surface lower-used features in onboarding.
2. Schedule push notifications around peak hours (8, 1, 2).
3. Target the least active age group ('45-54') with tailored content or incentives.
4. Fix UX issues on 'Messaging', the lowest-rated feature.
5. Optimise the experience for 'Web' devices, which have the shortest sessions.
